# Representaciones dispersas con Dictionary Learning y K-SVD

**Proyecto 1, Matematicas para la Computacion.**

Este notebook recorre el proceso completo, de la senal cruda al
diccionario entrenado. No reimplementa nada: todo lo importa de los
modulos de `src/`, que son los que se entregan. Aqui solo se explica
y se visualiza.

## El problema en una linea

Dada una senal `x`, encontrar un diccionario `D` y un vector de
coeficientes `alpha` con muy pocas entradas distintas de cero, tales
que `x = D alpha`.

Si `D` tiene 256 columnas y `alpha` solo 9 distintas de cero, una
senal de 128 muestras queda descrita con 9 numeros en vez de 128.

## Mapa del notebook

1. La senal de ECG y como se corta en ventanas
2. Por que el problema tiene infinitas soluciones
3. OMP: encontrar `alpha` con `D` fijo
4. K-SVD: encontrar `D`, y donde entra la descomposicion en valores
   singulares
5. Entrenamiento sobre el ECG real
6. Resultados

## 0. Los datos crudos

Antes de cualquier algoritmo, hay que ver con que se trabaja. Esta
seccion sigue los datos en tres etapas, con los numeros reales:

1. **La senal**: el ECG tal como llega, una tira de numeros.
2. **Las ventanas**: la tira cortada en pedazos de 128, uno por
   columna.
3. **`X`**: las ventanas centradas y normalizadas, que es lo que
   reciben OMP y K-SVD.

### 0.1 La senal

`cargar_ecg()` lee el registro 100 de MIT-BIH y devuelve dos cosas:
un arreglo de numpy con el voltaje en milivolts y la frecuencia de
muestreo. La primera vez lo descarga de PhysioNet a `proyecto1/datos/`,
unos 25 segundos; despues lo lee del disco en menos de un segundo. Es un arreglo de **una sola dimension**: no es una matriz
todavia.

In [6]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd

from Datos import cargar_ecg, construir_x, partir_en_ventanas
from Utils import TAM_VENTANA

np.set_printoptions(precision=3, suppress=True, linewidth=110)

senal, fs = cargar_ecg()

print(f"tipo:       {type(senal).__name__}, {senal.dtype}")
print(f"forma:      {senal.shape}  <- una dimension, 650 mil numeros")
print(f"frecuencia: {fs} muestras por segundo")
print(f"duracion:   {len(senal) / fs / 60:.1f} minutos")
print(f"rango:      de {senal.min():.3f} a {senal.max():.3f} mV")
print(f"promedio:   {senal.mean():.3f} mV")

tipo:       ndarray, float64
forma:      (650000,)  <- una dimension, 650 mil numeros
frecuencia: 360 muestras por segundo
duracion:   30.1 minutos
rango:      de -2.715 a 1.435 mV
promedio:   -0.306 mV


In [11]:
senal

array([-0.145, -0.145, -0.145, ..., -0.675, -0.765, -1.28 ], shape=(650000,))

Los primeros 20 numeros, cada uno con el instante en que se midio.
Entre una muestra y la siguiente pasan 1/360 de segundo, unos 2.8
milisegundos.

In [7]:
for i in range(20):
    print(f"muestra {i:>2}   t = {i / fs * 1000:6.2f} ms   "
          f"voltaje = {senal[i]:7.3f} mV")

muestra  0   t =   0.00 ms   voltaje =  -0.145 mV
muestra  1   t =   2.78 ms   voltaje =  -0.145 mV
muestra  2   t =   5.56 ms   voltaje =  -0.145 mV
muestra  3   t =   8.33 ms   voltaje =  -0.145 mV
muestra  4   t =  11.11 ms   voltaje =  -0.145 mV
muestra  5   t =  13.89 ms   voltaje =  -0.145 mV
muestra  6   t =  16.67 ms   voltaje =  -0.145 mV
muestra  7   t =  19.44 ms   voltaje =  -0.145 mV
muestra  8   t =  22.22 ms   voltaje =  -0.120 mV
muestra  9   t =  25.00 ms   voltaje =  -0.135 mV
muestra 10   t =  27.78 ms   voltaje =  -0.145 mV
muestra 11   t =  30.56 ms   voltaje =  -0.150 mV
muestra 12   t =  33.33 ms   voltaje =  -0.160 mV
muestra 13   t =  36.11 ms   voltaje =  -0.155 mV
muestra 14   t =  38.89 ms   voltaje =  -0.160 mV
muestra 15   t =  41.67 ms   voltaje =  -0.175 mV
muestra 16   t =  44.44 ms   voltaje =  -0.180 mV
muestra 17   t =  47.22 ms   voltaje =  -0.185 mV
muestra 18   t =  50.00 ms   voltaje =  -0.170 mV
muestra 19   t =  52.78 ms   voltaje =  -0.155 mV


Graficados, esos numeros dibujan el ECG. Cada punto es una muestra;
la linea solo los une. En tres segundos caben unos cuatro latidos, y
cada pico alto es el complejo QRS de uno.

In [8]:
n = 3 * fs
t = np.arange(n) / fs

fig = go.Figure(go.Scatter(x=t, y=senal[:n], mode="lines+markers",
                           line=dict(color="#52514e", width=1),
                           marker=dict(size=3, color="#2a78d6")))
fig.update_layout(title="Los primeros 3 segundos, muestra por muestra",
                  xaxis_title="segundos", yaxis_title="mV",
                  height=320, template="plotly_white",
                  showlegend=False)
fig.show()

### 0.2 Las ventanas

`partir_en_ventanas()` corta la tira en pedazos de 128 numeros
seguidos y pone **cada pedazo como una columna**. Las ventanas no se
enciman: la columna 0 es `senal[0:128]`, la 1 es `senal[128:256]`, y
asi. Las muestras que sobran al final se descartan.

In [9]:
crudas = partir_en_ventanas(senal)

df = pd.DataFrame(
    crudas,
    index=[f"muestra {i}" for i in range(crudas.shape[0])],
    columns=[f"ventana {j}" for j in range(crudas.shape[1])],
)

print(f"forma: {crudas.shape}  <- 128 filas, una columna por ventana")
print(f"{len(senal)} // {TAM_VENTANA} = {len(senal) // TAM_VENTANA} "
      f"ventanas completas")
print(f"cada ventana dura {TAM_VENTANA / fs * 1000:.0f} ms\n")

print("columna 0 == senal[0:128]:  ",
      np.array_equal(crudas[:, 0], senal[0:128]))
print("columna 1 == senal[128:256]:",
      np.array_equal(crudas[:, 1], senal[128:256]))

print("\nesquina superior izquierda, filas 0-9, columnas 0-5:")
print(crudas[:10, :6])
print("\nultimas 3 filas, columnas 0-5:")
print(crudas[-3:, :6])

forma: (128, 5078)  <- 128 filas, una columna por ventana
650000 // 128 = 5078 ventanas completas
cada ventana dura 356 ms

columna 0 == senal[0:128]:   True
columna 1 == senal[128:256]: True

esquina superior izquierda, filas 0-9, columnas 0-5:
[[-0.145 -0.33  -0.295 -0.385 -0.315 -0.38 ]
 [-0.145 -0.33  -0.31  -0.39  -0.305 -0.385]
 [-0.145 -0.335 -0.29  -0.38  -0.315 -0.38 ]
 [-0.145 -0.335 -0.295 -0.38  -0.315 -0.375]
 [-0.145 -0.34  -0.3   -0.38  -0.31  -0.355]
 [-0.145 -0.33  -0.305 -0.395 -0.295 -0.37 ]
 [-0.145 -0.34  -0.31  -0.385 -0.29  -0.39 ]
 [-0.145 -0.35  -0.325 -0.385 -0.305 -0.405]
 [-0.12  -0.355 -0.31  -0.385 -0.31  -0.41 ]
 [-0.135 -0.35  -0.3   -0.375 -0.32  -0.435]]

ultimas 3 filas, columnas 0-5:
[[-0.35  -0.285 -0.425 -0.325 -0.37  -0.395]
 [-0.325 -0.285 -0.39  -0.31  -0.365 -0.39 ]
 [-0.325 -0.29  -0.39  -0.305 -0.365 -0.375]]


In [10]:
df

,ventana 0,ventana 1,ventana 2,ventana 3,ventana 4,ventana 5,ventana 6,ventana 7,ventana 8,ventana 9,...,ventana 5068,ventana 5069,ventana 5070,ventana 5071,ventana 5072,ventana 5073,ventana 5074,ventana 5075,ventana 5076,ventana 5077
muestra 0,-0.145,-0.330,-0.295,-0.385,-0.315,-0.380,-0.360,-0.265,-0.390,-0.320,...,-0.360,-0.275,-0.330,-0.215,-0.355,-0.260,-0.490,-0.275,-0.350,-0.380
muestra 1,-0.145,-0.330,-0.310,-0.390,-0.305,-0.385,-0.340,-0.255,-0.390,-0.290,...,-0.355,-0.265,-0.330,-0.205,-0.365,-0.270,-0.505,-0.280,-0.190,-0.375
muestra 2,-0.145,-0.335,-0.290,-0.380,-0.315,-0.380,-0.335,-0.240,-0.365,-0.290,...,-0.355,-0.265,-0.310,-0.205,-0.415,-0.250,-0.540,-0.275,0.065,-0.385
muestra 3,-0.145,-0.335,-0.295,-0.380,-0.315,-0.375,-0.330,-0.260,-0.370,-0.300,...,-0.365,-0.265,-0.320,-0.215,-0.465,-0.240,-0.595,-0.280,0.375,-0.385
muestra 4,-0.145,-0.340,-0.300,-0.380,-0.310,-0.355,-0.335,-0.280,-0.375,-0.305,...,-0.380,-0.250,-0.340,-0.215,-0.470,-0.240,-0.565,-0.280,0.650,-0.395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
muestra 123,-0.330,-0.305,-0.475,-0.300,-0.375,-0.415,-0.275,-0.375,-0.330,-0.410,...,-0.305,-0.325,-0.225,-0.345,-0.285,-0.350,-0.265,-0.570,-0.360,-0.745
muestra 124,-0.340,-0.295,-0.455,-0.320,-0.380,-0.395,-0.280,-0.375,-0.335,-0.415,...,-0.315,-0.335,-0.220,-0.350,-0.280,-0.355,-0.275,-0.595,-0.335,-0.815
muestra 125,-0.350,-0.285,-0.425,-0.325,-0.370,-0.395,-0.275,-0.375,-0.335,-0.405,...,-0.315,-0.325,-0.220,-0.360,-0.265,-0.370,-0.290,-0.615,-0.340,-0.815
muestra 126,-0.325,-0.285,-0.390,-0.310,-0.365,-0.390,-0.280,-0.375,-0.340,-0.420,...,-0.305,-0.340,-0.215,-0.360,-0.255,-0.410,-0.285,-0.575,-0.340,-0.730


Casi todos los valores rondan -0.3 mV: es la linea base, donde el
corazon no esta haciendo nada visible. Para ver un latido dentro de la
matriz hay que buscar la columna donde esta el pico. En la columna 5
el pico cae en la fila 23.

In [5]:
col = 5
pico = int(np.argmax(crudas[:, col]))
print(f"columna {col}, pico en la fila {pico}\n")

for i in range(pico - 9, pico + 8):
    marca = "  <- pico R" if i == pico else ""
    print(f"fila {i:>3}   {crudas[i, col]:7.3f} mV{marca}")

columna 5, pico en la fila 23

fila  14    -0.570 mV
fila  15    -0.525 mV
fila  16    -0.405 mV
fila  17    -0.250 mV
fila  18    -0.090 mV
fila  19     0.120 mV
fila  20     0.410 mV
fila  21     0.690 mV
fila  22     0.885 mV
fila  23     0.960 mV  <- pico R
fila  24     0.850 mV
fila  25     0.520 mV
fila  26     0.050 mV
fila  27    -0.320 mV
fila  28    -0.500 mV
fila  29    -0.505 mV
fila  30    -0.445 mV


En unas diez filas, menos de 30 ms, el voltaje sube de -0.5 a casi
1 mV y vuelve a bajar. Eso es el complejo QRS: la contraccion de los
ventriculos.

### 0.3 La matriz X

`construir_x()` toma las mismas ventanas y a cada columna le hace dos
cosas:

```
1. centrar:     columna = columna - promedio(columna)
2. normalizar:  columna = columna / norma(columna)
```

La forma de la matriz no cambia, sigue siendo de 128 x 5078. Lo que
cambia es que cada columna queda con promedio 0 y norma 1.

In [6]:
x = construir_x(senal)

print(f"forma: {x.shape}  <- igual que las ventanas crudas\n")
print("esquina superior izquierda, filas 0-9, columnas 0-5:")
print(x[:10, :6])
print("\npromedio de las columnas 0-5:", x[:, :6].mean(axis=0))
print("norma de las columnas 0-5:   ",
      np.linalg.norm(x[:, :6], axis=0))

forma: (128, 5078)  <- igual que las ventanas crudas

esquina superior izquierda, filas 0-9, columnas 0-5:
[[ 0.034 -0.042 -0.018  0.01   0.004 -0.011]
 [ 0.034 -0.042 -0.023  0.     0.027 -0.012]
 [ 0.034 -0.053 -0.016  0.02   0.004 -0.011]
 [ 0.034 -0.053 -0.018  0.02   0.004 -0.009]
 [ 0.034 -0.064 -0.02   0.02   0.016 -0.002]
 [ 0.034 -0.042 -0.021 -0.01   0.05  -0.007]
 [ 0.034 -0.064 -0.023  0.01   0.061 -0.014]
 [ 0.034 -0.086 -0.028  0.01   0.027 -0.019]
 [ 0.044 -0.097 -0.023  0.01   0.016 -0.021]
 [ 0.038 -0.086 -0.02   0.03  -0.007 -0.03 ]]

promedio de las columnas 0-5: [-0.  0. -0. -0. -0. -0.]
norma de las columnas 0-5:    [1. 1. 1. 1. 1. 1.]


El pico de la columna 5, calculado a mano. Los numeros cambian, pero
la forma se conserva: el pico sigue en la fila 23.

In [7]:
media = crudas[:, col].mean()
norma = np.linalg.norm(crudas[:, col] - media)

print(f"promedio de la columna {col} cruda:  {media:.4f}")
print(f"norma de la columna ya centrada:  {norma:.4f}")
print(f"X[{pico}, {col}] = ({crudas[pico, col]:.3f} - ({media:.4f})) "
      f"/ {norma:.4f} = {(crudas[pico, col] - media) / norma:.3f}")
print(f"valor en X:                       {x[pico, col]:.3f}")

promedio de la columna 5 cruda:  -0.3494
norma de la columna ya centrada:  2.8586
X[23, 5] = (0.960 - (-0.3494)) / 2.8586 = 0.458
valor en X:                       0.458


### 0.4 La forma de la matriz completa

Un mapa de calor de las primeras 60 columnas de `X`. Cada columna
vertical es una ventana; el color es el valor de cada fila. Las
franjas brillantes son los picos QRS: aparecen en filas distintas
porque los latidos no caen en el mismo lugar de cada ventana. Esa
variacion es justo la que el diccionario tiene que aprender a cubrir.

In [8]:
fig = go.Figure(go.Heatmap(z=x[:, :60], colorscale="RdBu_r",
                           zmid=0, colorbar=dict(title="valor")))
fig.update_layout(title="X, primeras 60 columnas (ventanas)",
                  xaxis_title="columna = ventana",
                  yaxis_title="fila = muestra dentro de la ventana",
                  yaxis_autorange="reversed",
                  height=420, template="plotly_white")
fig.show()

### 0.5 Varios pacientes

Todo lo anterior es un solo paciente, el registro 100. MIT-BIH tiene
48 registros, y `construir_x_registros()` arma `X` con los que se le
pidan. Cada registro se ventanea por separado y las matrices se pegan
por columnas, asi que la forma sigue siendo de 128 filas; solo crecen
las columnas.

Ademas de `X` devuelve `origen`: de que registro salio cada columna.
`partir_entrenamiento` lo usa para cortar a la mitad **dentro de cada
paciente**, y que todos queden representados en entrenamiento y en
prueba.

Con `TODOS = True` se cargan los 48. La primera vez tarda unos 20
minutos porque descarga los 48 registros, unos 100 MB; despues se
leen del disco. `X` pasa a unas 244 mil columnas. Para entrenar con
todos desde la terminal:

```
python Experimento.py --todos
python Experimento.py --registros 100 101 103
```

In [9]:
from Datos import construir_x_registros
from Utils import REGISTROS_MITDB

TODOS = False
registros = REGISTROS_MITDB if TODOS else ("100", "101", "103")

x_varios, origen, _ = construir_x_registros(registros)

print(f"{len(registros)} registros, X de {x_varios.shape[0]} x "
      f"{x_varios.shape[1]}\n")
for registro in registros:
    print(f"  registro {registro}: {(origen == registro).sum()} ventanas")

3 registros, X de 128 x 15234

  registro 100: 5078 ventanas
  registro 101: 5078 ventanas
  registro 103: 5078 ventanas


## Conceptos: senal, atomo, diccionario y SVD

Con los datos a la vista, conviene fijar el vocabulario, porque todo
el notebook habla en estos terminos.

### Senal

Para el algoritmo, una senal es **una columna de `X`**: una ventana de
128 muestras seguidas del ECG, centrada y normalizada. `X` tiene una
senal por columna, unas 5000 en total.

```
X  =  [ x_1  x_2  x_3  ...  x_n ]      128 x n
         ^
         una senal: 128 numeros
```

### Atomo

Un atomo es **una forma de onda basica**: un vector de 128 numeros,
del mismo tamano que una senal, con norma 1. Por si solo no es ningun
pedazo real del ECG. Es una pieza con la que se arman pedazos: un pico
angosto, una onda suave, una bajada.

La norma 1 importa: asi el atomo solo aporta **forma** y el tamano lo
pone su coeficiente. Si los atomos tuvieran normas distintas, OMP
preferiria a los mas largos por ser largos, no por parecerse mas.

### Diccionario

El diccionario `D` es **la coleccion de todos los atomos**, puestos
como columnas:

```
D  =  [ d_1  d_2  d_3  ...  d_256 ]    128 x 256
```

Cada senal se reconstruye como una suma pesada de **unos pocos**
atomos:

```
x  ~  alpha_1 d_1 + alpha_2 d_2 + ... + alpha_256 d_256  =  D alpha
```

y casi todos los `alpha_i` valen cero. Por ejemplo:

```
x_37  ~  0.8 d_12  +  0.3 d_140  -  0.1 d_7
```

Tres atomos de 256. La columna 37 de `alpha` tiene 253 ceros. A eso
se le llama **representacion dispersa**.

El nombre viene de la analogia con un diccionario de palabras: con un
vocabulario fijo se escriben muchisimas oraciones distintas, y cada
oracion usa pocas palabras. Aqui el vocabulario son formas de onda y
las oraciones son ventanas de ECG.

**Por que se aprende y no se fija.** Existen diccionarios fijos, como
los senos y cosenos de Fourier o las wavelets. Sirven para cualquier
senal, y justo por eso no son los mejores para ninguna en particular.
Un diccionario aprendido de ECGs tiene atomos con forma de latido, y
por eso necesita menos atomos para reconstruir un latido.

### SVD: descomposicion en valores singulares

Es la herramienta que usa K-SVD para actualizar cada atomo, asi que
vale la pena entenderla bien.

**Que es.** Cualquier matriz `A` de `m x n`, cuadrada o no, se puede
escribir como el producto de tres:

```
A  =  U  Sigma  V'

U      m x m   ortogonal: sus columnas u_1, u_2, ... miden 1 y son
               perpendiculares entre si
Sigma  m x n   ceros fuera de la diagonal; en la diagonal los valores
               singulares sigma_1 >= sigma_2 >= ... >= 0
V      n x n   ortogonal: columnas v_1, v_2, ...
```

Que una matriz sea ortogonal quiere decir que su inversa es su
transpuesta, `U' U = I`, y que no cambia longitudes ni angulos: es una
rotacion o una reflexion.

**Que significa geometricamente.** Multiplicar por `A` es hacer tres
cosas en orden, leyendo `U Sigma V' x` de derecha a izquierda:

1. `V'` **gira** el vector.
2. `Sigma` **estira** cada eje por su `sigma_i`.
3. `U` **gira** otra vez el resultado.

Los giros no deforman nada, asi que toda la deformacion de `A` esta en
`Sigma`. Un circulo de radio 1 sale convertido en una elipse cuyos
semiejes miden `sigma_1`, `sigma_2`, ... y apuntan hacia `u_1`, `u_2`,
...

**Como se calcula a mano.**

1. Se forma `A' A`, que es cuadrada de `n x n` y simetrica.
2. Se sacan sus eigenvalores `lambda_i` y se ordenan de mayor a menor.
   Siempre son `>= 0`, porque `lambda = |A v|^2` cuando `v` mide 1.
3. Valores singulares: `sigma_i = sqrt(lambda_i)`.
4. Los eigenvectores de `A' A`, normalizados a norma 1, son las
   columnas `v_i` de `V`.
5. Las columnas de `U` salen de `u_i = A v_i / sigma_i`. Calcularlas
   asi, y no por separado, amarra el signo de cada `u_i` al de su
   `v_i`.

**Ejemplo.** Con

```
A = [ 1  0 ]
    [ 0  1 ]
    [ 1  1 ]
```

- `A' A = [[2, 1], [1, 2]]`. Su polinomio es
  `(2 - lambda)^2 - 1 = (lambda - 3)(lambda - 1)`, asi que
  `lambda_1 = 3` y `lambda_2 = 1`.
- `sigma_1 = sqrt(3) = 1.732` y `sigma_2 = 1`.
- Para `lambda = 3`: `(A'A - 3I) v = 0` da `y = x`, o sea
  `v_1 = (1, 1) / sqrt(2)`.
  Para `lambda = 1`: da `y = -x`, o sea `v_2 = (1, -1) / sqrt(2)`.
- `u_1 = A v_1 / sqrt(3) = (1, 1, 2) / sqrt(6)` y
  `u_2 = A v_2 / 1 = (1, -1, 0) / sqrt(2)`.

**La SVD como suma de piezas de rango 1.** Multiplicando por bloques,
la descomposicion tambien se escribe como

```
A  =  sigma_1 u_1 v_1'  +  sigma_2 u_2 v_2'  +  ...
```

Cada termino `u_i v_i'` es una columna por un renglon: una matriz de
**rango 1**. Como los `sigma_i` vienen ordenados, el primer termino es
el que mas pesa, el segundo el que sigue, y asi.

**Teorema de Eckart-Young.** Si se conservan solo los primeros `k`
terminos, el resultado es **la mejor aproximacion de rango `k`** de
`A`: ninguna otra matriz de rango `k` queda mas cerca. Lo que se
pierde son justo los `sigma` que se tiraron:

```
|| A - A_k ||^2  =  sigma_{k+1}^2 + sigma_{k+2}^2 + ...
```

En el ejemplo, quedarse con `k = 1` da

```
A_1 = sigma_1 u_1 v_1' = 0.5 * [ 1  1 ]
                               [ 1  1 ]
                               [ 2  2 ]
```

y el error al cuadrado es `sigma_2^2 = 1` de un total de
`sigma_1^2 + sigma_2^2 = 4`: con una sola pieza se conserva el 75% de
la energia de `A`.

**Por que le importa a K-SVD.** Al actualizar el atomo `j` se tiene un
residual `R`: lo que las senales que usan ese atomo no logran explicar
con los demas. Se busca el atomo `d_j` y sus coeficientes `alpha_j`
que mejor lo expliquen. El producto `d_j alpha_j'` es columna por
renglon, de rango 1, asi que la respuesta es el primer termino de la
SVD de `R`:

```
d_j      =  u_1
alpha_j  =  sigma_1 v_1'
```

`u_1` ya mide 1, asi que el atomo nuevo queda normalizado sin hacer
nada mas. Y `sigma_1^2` entre la suma de todos los `sigma_i^2` dice
que fraccion del residual logra capturar el atomo. Esa fraccion es la
que se grafica en la seccion 4.

La celda siguiente comprueba el ejemplo con numpy.

In [ ]:
import numpy as np

a = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
u, sigma, vt = np.linalg.svd(a)

print("valores singulares:", np.round(sigma, 4), " (sqrt(3) y 1)")
# numpy puede devolver u_1 y v_1 con el signo al reves que el
# ejemplo. Es la misma solucion: cambiar el signo de los dos a la
# vez no altera sigma_1 u_1 v_1'
print("v_1 =", np.round(vt[0], 4), "  u_1 =", np.round(u[:, 0], 4))

# Reconstruccion completa: U Sigma V' devuelve A
s = np.zeros_like(a)
s[:2, :2] = np.diag(sigma)
print("U Sigma V' = A:", np.allclose(u @ s @ vt, a))

# Mejor aproximacion de rango 1 y la energia que conserva
a1 = sigma[0] * np.outer(u[:, 0], vt[0])
print("A_1 =\n", np.round(a1, 4))
print(f"energia conservada: {sigma[0]**2 / (sigma**2).sum():.0%}")

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from Datos import cargar_ecg, construir_x, partir_en_ventanas
from Experimento import ARCHIVO_MODELO, curva_error, partir_entrenamiento
from Figuras import elegir_ventana
from Ksvd import actualizar_atomo, diccionario_inicial, ksvd
from Omp import omp, omp_matriz
from Utils import (ERROR_OMP, FRECUENCIA_HZ, ITERACIONES_KSVD,
                   MAX_ATOMOS_OMP, N_ATOMOS, SEMILLA, TAM_VENTANA,
                   dispersion_media, error_relativo,
                   normalizar_columnas)

# La misma paleta validada que usan las figuras del reporte
TINTA = "#52514e"
REJILLA = "#e8e7e2"
AZUL = "#2a78d6"
NARANJA = "#eb6834"
AQUA = "#1baf7a"

PLANTILLA = dict(
    template="plotly_white",
    font=dict(size=12, color=TINTA),
    margin=dict(l=60, r=30, t=60, b=50),
)

print(f"ventana {TAM_VENTANA} muestras | {N_ATOMOS} atomos | "
      f"error objetivo {ERROR_OMP} | tope {MAX_ATOMOS_OMP} atomos")

ventana 128 muestras | 256 atomos | error objetivo 0.1 | tope 16 atomos


## 1. La senal

Se usa el registro 100 de **MIT-BIH Arrhythmia**, de PhysioNet. Son
30 minutos de ECG muestreados a 360 Hz. `wfdb` lo entrega ya convertido
a milivolts.

In [2]:
senal, fs = cargar_ecg()

print(f"{len(senal)} muestras a {fs} Hz = "
      f"{len(senal) / fs / 60:.1f} minutos")

# Diez segundos, que es donde se alcanzan a contar los latidos
tramo = slice(0, 10 * fs)
t = np.arange(len(senal))[tramo] / fs

fig = go.Figure(go.Scatter(x=t, y=senal[tramo], mode="lines",
                           line=dict(color=TINTA, width=1.5),
                           name="ECG"))
fig.update_layout(title="Registro 100, primeros 10 segundos",
                  xaxis_title="segundos", yaxis_title="mV",
                  height=320, showlegend=False, **PLANTILLA)
fig.show()

650000 muestras a 360 Hz = 30.1 minutos


### Por que se corta en ventanas

A K-SVD no se le pasa una senal de 650 mil muestras. Se corta en
ventanas de `TAM_VENTANA = 128` muestras, que a 360 Hz son 356 ms,
poco mas que un latido. Cada ventana es una columna de la matriz `X`.

A cada ventana se le hacen dos cosas antes de guardarla:

- **Se le quita su media.** El ECG trae deriva de linea base, un
  vaiven lento del electrodo. Sin quitarla, el primer atomo que
  aprende el diccionario es una constante, que no describe ninguna
  forma de onda.
- **Se normaliza a norma 1.** Asi lo unico que distingue a una
  columna de otra es la forma de onda, no su amplitud.

In [3]:
crudas = partir_en_ventanas(senal)
x = construir_x(senal)

print(f"X de {x.shape[0]} x {x.shape[1]}")
print(f"normas de columna: min {np.linalg.norm(x, axis=0).min():.4f} "
      f"max {np.linalg.norm(x, axis=0).max():.4f}")

ms = np.arange(TAM_VENTANA) / fs * 1000
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Ventanas crudas",
                                    "Centradas y normalizadas"))

for j in range(0, 40, 4):
    fig.add_trace(go.Scatter(x=ms, y=crudas[:, j], mode="lines",
                             line=dict(color=TINTA, width=1),
                             opacity=0.45, showlegend=False),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=ms, y=x[:, j], mode="lines",
                             line=dict(color=AZUL, width=1),
                             opacity=0.45, showlegend=False),
                  row=1, col=2)

fig.update_xaxes(title_text="ms")
fig.update_layout(height=340,
                  title="Diez ventanas, antes y despues del "
                        "preprocesamiento", **PLANTILLA)
fig.show()

X de 128 x 5078
normas de columna: min 1.0000 max 1.0000


## 2. Por que el problema tiene infinitas soluciones

El diccionario es **sobrecompleto**: tiene mas columnas que filas.
Con `D` de 128 x 256, el sistema

```
x = D alpha
```

tiene 128 ecuaciones y 256 incognitas. Esta subdeterminado, asi que
no tiene una solucion sino un subespacio afin de soluciones de
dimension 128.

Eso no es un defecto, es el diseno. De todas esas soluciones se busca
la **mas dispersa**, la que usa menos atomos. Un diccionario cuadrado
y de rango completo daria una sola solucion, y no habria nada que
elegir.

El problema es que buscar la solucion mas dispersa es NP-dificil. Por
eso se usa un metodo voraz, OMP, que no garantiza el optimo pero en
la practica se le acerca.

In [4]:
d_inicial = diccionario_inicial(x)
print(f"D: {d_inicial.shape[0]} x {d_inicial.shape[1]}")
print(f"ecuaciones: {d_inicial.shape[0]}, "
      f"incognitas: {d_inicial.shape[1]}")
print(f"dimension del espacio de soluciones: "
      f"{d_inicial.shape[1] - np.linalg.matrix_rank(d_inicial)}")

D: 128 x 256
ecuaciones: 128, incognitas: 256
dimension del espacio de soluciones: 129


## 3. OMP: encontrar `alpha` con `D` fijo

Orthogonal Matching Pursuit es voraz. En cada vuelta:

1. Escoge el atomo **mas correlacionado con el residual**, es decir el
   de mayor `|d_i . r|`.
2. Resuelve minimos cuadrados sobre **todos** los atomos escogidos
   hasta el momento, no solo el nuevo.
3. Actualiza el residual.
4. Para cuando el error relativo baja de `ERROR_OMP` o se agota el
   presupuesto `MAX_ATOMOS_OMP`.

El paso 2 es el que le da el nombre: resolver sobre todo el conjunto
deja el residual **ortogonal** al espacio generado por los atomos
elegidos, y por eso ninguno se vuelve a escoger.

### Dos trampas de la notacion de las slides

- `alpha = D' x` **no** da la representacion dispersa. Solo invierte
  `x = D alpha` cuando `D` es ortogonal, y aqui es sobrecompleto a
  proposito. Da las correlaciones, que son el punto de partida del
  paso 1, no la solucion.
- `(D_I)^-1` es una **pseudoinversa**. `D_I` son los atomos
  seleccionados, una matriz alta y rectangular que no tiene inversa.
  Ahi se resuelve un problema de minimos cuadrados.

### OMP vuelta por vuelta

Se toma una ventana con un latido completo y se corre OMP con un tope
creciente de atomos, para ver como va bajando el residual.

In [5]:
col = elegir_ventana(x)
objetivo = x[:, col]

topes = range(1, 13)
errores, reconstrucciones = [], []

for tope in topes:
    coef, elegidos = omp(d_inicial, objetivo, error=0.0,
                         max_atomos=tope)
    recon = d_inicial @ coef
    reconstrucciones.append(recon)
    errores.append(error_relativo(objetivo, recon))

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.55, 0.45],
    subplot_titles=("Reconstruccion acumulada",
                    "Error relativo por atomo agregado"))

fig.add_trace(go.Scatter(x=ms, y=objetivo, mode="lines",
                         name="original",
                         line=dict(color=TINTA, width=3)),
              row=1, col=1)
for tope in (1, 2, 4, 8, 12):
    fig.add_trace(go.Scatter(x=ms, y=reconstrucciones[tope - 1],
                             mode="lines", name=f"{tope} atomos",
                             line=dict(width=1.6)),
                  row=1, col=1)

fig.add_trace(go.Scatter(x=list(topes), y=errores, mode="lines+markers",
                         line=dict(color=AZUL, width=2),
                         marker=dict(size=8), showlegend=False),
              row=1, col=2)
fig.add_hline(y=ERROR_OMP, line=dict(color=NARANJA, dash="dash"),
              annotation_text=f"ERROR_OMP = {ERROR_OMP}",
              annotation_position="top right", row=1, col=2)

fig.update_xaxes(title_text="ms", row=1, col=1)
fig.update_xaxes(title_text="atomos usados", row=1, col=2)
fig.update_yaxes(title_text="error relativo", row=1, col=2)
fig.update_layout(height=400,
                  title="OMP sobre una ventana, con el diccionario "
                        "sin entrenar", **PLANTILLA)
fig.show()

print(f"error con 1 atomo: {errores[0]:.4f}")
print(f"error con 12 atomos: {errores[-1]:.4f}")

error con 1 atomo: 0.2472
error con 12 atomos: 0.0562


### Validacion de OMP

Antes de confiar en OMP hay que probarlo donde **si se sabe la
respuesta**: se arma un diccionario al azar, se generan senales
usando un conjunto conocido de atomos, y se revisa si OMP recupera
justo esos. Si esto falla, cualquier problema de K-SVD es consecuencia
y no causa.

In [6]:
rng = np.random.default_rng(SEMILLA)
k_p, m_p, n_p, usados = 40, 100, 300, 5

d_prueba = normalizar_columnas(rng.standard_normal((k_p, m_p)))

aciertos = exactos = 0
for _ in range(n_p):
    verdaderos = rng.choice(m_p, usados, replace=False)
    sintetica = d_prueba[:, verdaderos] @ rng.standard_normal(usados)
    _, elegidos = omp(d_prueba, sintetica, error=1e-6,
                      max_atomos=usados)
    comunes = len(set(elegidos) & set(verdaderos))
    aciertos += comunes
    exactos += comunes == usados

print(f"diccionario {k_p} x {m_p}, {n_p} senales de {usados} atomos")
print(f"  atomos recuperados:        {aciertos / (n_p * usados):.2%}")
print(f"  senales con soporte exacto: {exactos / n_p:.2%}")

diccionario 40 x 100, 300 senales de 5 atomos
  atomos recuperados:        99.87%
  senales con soporte exacto: 99.33%


## 4. K-SVD: encontrar `D`

K-SVD alterna dos pasos, igual que k-means:

- Con `D` fijo, calcula todas las dispersiones con OMP.
- Con las dispersiones fijas, actualiza `D` **un atomo a la vez**.

### La actualizacion de un atomo, que es donde entra la SVD

Para actualizar el atomo `j`:

1. Se toman solo las senales `w` que **si lo usan**, o sea donde
   `alpha[j, l] != 0`.
2. Se arma el residual `R = X_w - D alpha_w`, con la fila `j` de
   `alpha_w` puesta a cero. Es decir: que queda sin explicar de esas
   senales si se le quita la contribucion del atomo `j`.
3. Se busca el atomo `d_j` y los coeficientes `alpha_j` que mejor
   expliquen `R`.

Y aqui esta el punto. El producto `d_j alpha_j'` es columna por
renglon, o sea una **matriz de rango 1**. Preguntar cual es el mejor
atomo equivale a preguntar cual es la mejor aproximacion de rango 1
de `R`, y eso lo contesta el teorema de **Eckart-Young**:

```
R ~= sigma_1 u_1 v_1'   ->   d_j = u_1 ,   alpha_j = sigma_1 v_1
```

Una SVD por atomo y por iteracion. De ahi el nombre del algoritmo.

**Por que solo las senales de `w`.** Si la actualizacion usara todas
las senales, el atomo entraria en senales que no lo estaban usando y
se destruiria la dispersion que OMP acababa de construir. Restringir
a `w` conserva el patron de ceros.

### Viendo la aproximacion de rango 1

Se toma un atomo concreto, se arma su residual y se compara contra su
mejor aproximacion de rango 1. Si el primer valor singular domina a
los demas, el residual casi es de rango 1 y la actualizacion captura
casi todo.

In [7]:
entrena, prueba = partir_entrenamiento(x)

d_demo = diccionario_inicial(entrena)
alpha_demo = omp_matriz(d_demo, entrena[:, :600])

# Un atomo que use bastantes senales, para que el residual tenga
# suficientes columnas
usos = np.count_nonzero(alpha_demo, axis=1)
j = int(np.argmax(usos))
w = np.flatnonzero(alpha_demo[j, :])

coeficientes = alpha_demo[:, w].copy()
coeficientes[j, :] = 0.0
# w indexa dentro del subconjunto de 600 con el que se calculo
# alpha_demo, asi que el residual se arma sobre ese mismo subconjunto
residual = entrena[:, :600][:, w] - d_demo @ coeficientes

u, sigma, vt = np.linalg.svd(residual, full_matrices=False)
energia = sigma ** 2 / (sigma ** 2).sum()

print(f"atomo {j}, usado por {len(w)} senales")
print(f"residual de {residual.shape[0]} x {residual.shape[1]}")
print(f"energia del primer valor singular: {energia[0]:.1%}")

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=("Espectro del residual",
                                    "Atomo viejo contra atomo nuevo"))

fig.add_trace(go.Bar(x=list(range(1, 16)), y=energia[:15],
                     marker_color=AZUL, showlegend=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=ms, y=d_demo[:, j], mode="lines",
                         name="atomo antes",
                         line=dict(color=TINTA, width=2)),
              row=1, col=2)
fig.add_trace(go.Scatter(x=ms, y=u[:, 0], mode="lines",
                         name="u_1, atomo despues",
                         line=dict(color=AZUL, width=2)),
              row=1, col=2)

fig.update_xaxes(title_text="valor singular", row=1, col=1)
fig.update_yaxes(title_text="fraccion de energia", row=1, col=1)
fig.update_xaxes(title_text="ms", row=1, col=2)
fig.update_layout(height=380,
                  title="La actualizacion de un atomo es una "
                        "aproximacion de rango 1", **PLANTILLA)
fig.show()

atomo 50, usado por 130 senales
residual de 128 x 130
energia del primer valor singular: 77.8%


### Validacion de K-SVD

La misma idea que con OMP: se genera un diccionario conocido, se
crean senales a partir de el, y se revisa si K-SVD lo recupera. La
medida es el coseno entre cada atomo verdadero y el aprendido mas
parecido. El valor absoluto porque un atomo invertido representa
igual de bien.

In [8]:
rng = np.random.default_rng(SEMILLA)
k_v, m_v, n_v, usados_v = 20, 50, 1500, 3

verdadero = normalizar_columnas(rng.standard_normal((k_v, m_v)))
sinteticas = normalizar_columnas(np.column_stack([
    verdadero[:, rng.choice(m_v, usados_v, replace=False)]
    @ rng.standard_normal(usados_v) for _ in range(n_v)]))

d_rec, _, hist_rec = ksvd(sinteticas, m=m_v, iteraciones=40,
                          error=1e-6, max_atomos=usados_v,
                          verboso=False)

parecido = np.abs(verdadero.T @ d_rec).max(axis=1)
err_rec = [h["error"] for h in hist_rec]

print(f"error {err_rec[0]:.4f} -> {err_rec[-1]:.4f}")
print(f"atomos con coseno > 0.99: {(parecido > 0.99).sum()}/{m_v}")
print(f"parecido promedio: {parecido.mean():.4f}")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Error por iteracion",
                                    "Parecido con el atomo verdadero"))
fig.add_trace(go.Scatter(y=err_rec, mode="lines",
                         line=dict(color=AZUL, width=2),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=parecido, nbinsx=25,
                           marker_color=AQUA, showlegend=False),
              row=1, col=2)
fig.update_xaxes(title_text="iteracion", row=1, col=1)
fig.update_xaxes(title_text="coseno", row=1, col=2)
fig.update_layout(height=360,
                  title="K-SVD recupera el diccionario que genero "
                        "los datos", **PLANTILLA)
fig.show()

error 0.4543 -> 0.1603
atomos con coseno > 0.99: 42/50
parecido promedio: 0.9763


El error no baja de forma estrictamente monotona, y eso **no es un
error de programacion**. OMP es voraz y no garantiza la solucion
optima, asi que de vez en cuando una iteracion sale peor que la
anterior.

## 5. Entrenamiento sobre el ECG real

Dos decisiones del montaje:

- **La particion es temporal, no aleatoria.** Se entrena con la
  primera mitad de las ventanas y se evalua con la segunda. Las
  ventanas contiguas de un ECG se parecen tanto entre si que
  repartirlas al azar dejaria en prueba vecinas directas de las de
  entrenamiento, lo que inflaria el resultado.
- **`D_0` sale de las propias senales**, no de ruido. Arrancar con
  columnas de `X` pone a K-SVD en la zona correcta del espacio desde
  el principio.

El entrenamiento tarda unos minutos, asi que si ya existe
`out/modelo.npz` se carga en vez de repetirlo.

In [9]:
import os

if os.path.exists(ARCHIVO_MODELO):
    datos = np.load(ARCHIVO_MODELO)
    d_ecg = datos["diccionario"]
    alpha_prueba = datos["alpha_prueba"]
    dispersion = datos["historia_dispersion"]
    print(f"cargado de {ARCHIVO_MODELO}")
else:
    d_ecg, alpha_entrena, historia = ksvd(entrena)
    alpha_prueba = omp_matriz(d_ecg, prueba)
    dispersion = np.array([h["dispersion"] for h in historia])

print(f"\nD de {d_ecg.shape[0]} x {d_ecg.shape[1]}")
print(f"error en prueba: {error_relativo(prueba, d_ecg @ alpha_prueba):.4f}")
print(f"atomos por senal: {dispersion_media(alpha_prueba):.2f}")

cargado de ../out/modelo.npz

D de 128 x 256
error en prueba: 0.1148
atomos por senal: 9.46


### Que mejora durante el entrenamiento

Aqui hay un detalle que confunde si no se dice: **el error de
reconstruccion no baja con las iteraciones**. Se queda clavado en
0.10 porque ese *es* el criterio de paro de OMP: en cuanto llega al
10% de error, deja de agregar atomos.

Lo que K-SVD mejora es la **dispersion**: cuantos atomos hacen falta
para alcanzar ese mismo error.

In [10]:
fig = go.Figure(go.Scatter(
    x=np.arange(1, len(dispersion) + 1), y=dispersion,
    mode="lines+markers", line=dict(color=AZUL, width=2),
    marker=dict(size=7), showlegend=False))
fig.update_layout(
    title="Atomos por senal a lo largo del entrenamiento",
    xaxis_title="iteracion de K-SVD",
    yaxis_title="atomos por senal", height=340, **PLANTILLA)
fig.show()

print(f"de {dispersion[0]:.2f} a {dispersion[-1]:.2f} atomos "
      f"para el mismo error")

de 10.62 a 8.42 atomos para el mismo error


## 6. Resultados

### Que aprendio el diccionario

Los atomos ordenados por uso, o sea por en cuantas senales participa
cada uno. Si el entrenamiento sirvio, aqui no hay ruido: hay
complejos QRS en distintas posiciones y polaridades.

In [11]:
usos_ecg = np.count_nonzero(alpha_prueba, axis=1)
orden = np.argsort(-usos_ecg)[:18]

fig = make_subplots(
    rows=3, cols=6,
    subplot_titles=[f"{usos_ecg[j]} senales" for j in orden],
    vertical_spacing=0.12, horizontal_spacing=0.03)

for i, j in enumerate(orden):
    fig.add_trace(go.Scatter(y=d_ecg[:, j], mode="lines",
                             line=dict(color=AZUL, width=1.5),
                             showlegend=False),
                  row=i // 6 + 1, col=i % 6 + 1)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_annotations(font_size=10)
fig.update_layout(height=520,
                  title="Los 18 atomos mas usados", **PLANTILLA)
fig.show()

### Original contra reconstruccion

El entregable que pide el enunciado. Los trazos van desplazados en
vertical porque encimados se tapan entre si.

In [12]:
col_p = elegir_ventana(prueba)
objetivo_p = prueba[:, col_p]
paso = (objetivo_p.max() - objetivo_p.min()) * 0.9
topes_p = (2, 4, 8, 16)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ms, y=objetivo_p, mode="lines",
                         name="original",
                         line=dict(color=TINTA, width=3)))

for i, tope in enumerate(topes_p, start=1):
    coef, _ = omp(d_ecg, objetivo_p, error=0.0, max_atomos=tope)
    recon = d_ecg @ coef
    err = error_relativo(objetivo_p, recon)
    fig.add_trace(go.Scatter(
        x=ms, y=recon - i * paso, mode="lines",
        name=f"{tope} atomos, error {err:.3f}",
        line=dict(width=2)))

fig.update_layout(
    title="Un latido y sus reconstrucciones",
    xaxis_title="ms", yaxis=dict(showticklabels=False),
    height=480, **PLANTILLA)
fig.show()

### Cuanto cuesta cada nivel de fidelidad

La curva de error contra el tope de atomos, medida sobre el conjunto
de prueba.

In [13]:
curva = curva_error(d_ecg, prueba[:, :400])
permitidos = [f["permitidos"] for f in curva]
errs = [f["error"] for f in curva]

fig = go.Figure(go.Scatter(
    x=permitidos, y=errs, mode="lines+markers",
    line=dict(color=AZUL, width=2), marker=dict(size=9),
    hovertemplate="%{x} atomos<br>error %{y:.4f}<extra></extra>",
    showlegend=False))
fig.update_layout(
    title="Error de reconstruccion segun el tope de atomos",
    xaxis_title="atomos permitidos por senal",
    yaxis_title="error relativo", height=380, **PLANTILLA)
fig.show()

for fila in curva:
    print(f"{fila['permitidos']:>3} atomos  ->  "
          f"error {fila['error']:.4f}")

  1 atomos  ->  error 0.3072
  2 atomos  ->  error 0.2180
  4 atomos  ->  error 0.1633
  6 atomos  ->  error 0.1384
  8 atomos  ->  error 0.1222
 10 atomos  ->  error 0.1116
 12 atomos  ->  error 0.1044
 16 atomos  ->  error 0.0950
 24 atomos  ->  error 0.0848
 32 atomos  ->  error 0.0778


## Conclusiones

**La implementacion funciona y esta verificada contra casos donde se
conoce la respuesta.** OMP recupera el soporte exacto en 99 de cada
100 senales sinteticas, y K-SVD reconstruye el diccionario que genero
los datos con coseno mayor a 0.99 en la mayoria de los atomos.

**Sobre el ECG, el diccionario aprende complejos QRS.** No es una
observacion cosmetica: significa que la base entrenada capturo la
estructura de la senal en lugar de memorizar ventanas.

**La compresion es real.** Una ventana de 128 muestras se reconstruye
al 10% de error con unos 9 atomos, y al 3% con 16. El costo de
guardar son los indices y los coeficientes, no las 128 muestras.

**Lo que mejora con el entrenamiento es la dispersion, no el error.**
Con un criterio de paro por error fijo, K-SVD no baja el error, baja
cuantos atomos hacen falta para alcanzarlo: de 10.62 a 8.42.

**La brecha entre entrenamiento y prueba es chica**, de 0.1010 a
0.1148, lo que indica que el diccionario generaliza y no memorizo las
ventanas con las que se entreno.
